# Домашняя работа 3

**ФИО и группа:** Кулик Георгий, Т26ОАД-Т.МО31.

## Подготовка

In [ ]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

## Исходники домашней работы

In [ ]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei": "string"})
lookup = organizations[org_keys + ["organization_name"]].copy()
assert not lookup.duplicated(org_keys).any()
print(organizations.head(3))

## H1 Профиль отказов

In [ ]:
prepared = decisions.copy()
prepared["is_denied"] = prepared["action_taken"].eq(3)
denial = prepared.groupby(org_keys).agg(
    n_decisions=("action_taken", "size"),
    n_denied=("is_denied", "sum"),
).reset_index()
denial["denial_rate"] = denial["n_denied"] / denial["n_decisions"]
home_denial_profile = denial.merge(
    lookup, on=org_keys, how="left", validate="one_to_one", indicator=True
)
assert len(home_denial_profile) == len(denial)
assert home_denial_profile["_merge"].eq("both").all()
assert home_denial_profile["n_decisions"].sum() == len(decisions)
assert home_denial_profile["n_denied"].sum() == decisions["action_taken"].eq(3).sum()
assert home_denial_profile["denial_rate"].between(0, 1).all()
home_denial_profile = home_denial_profile.drop(columns="_merge")
home_denial_profile.to_csv(WORK / "home_denial_profile.csv", index=False)
print("Организаций:", len(home_denial_profile))
print("Решений:", home_denial_profile["n_decisions"].sum())
print("Отказов:", home_denial_profile["n_denied"].sum())
print("Общая доля отказов:", home_denial_profile["n_denied"].sum() / home_denial_profile["n_decisions"].sum())
home_denial_profile.head()

## H2 Группы для просмотра

In [ ]:
large_groups = home_denial_profile.loc[
    home_denial_profile["n_decisions"].ge(20)
].copy()
large_groups.to_csv(WORK / "home_denial_groups20.csv", index=False)
large_total = large_groups["n_decisions"].sum()
large_denied = large_groups["n_denied"].sum()
print("Групп с 20+ решениями:", len(large_groups))
print("Решений в группах:", large_total)
print("Отказов в группах:", large_denied)
print("Общая доля отказов выбранных групп:", large_denied / large_total)
print("Простое среднее долей отказов организаций:", large_groups["denial_rate"].mean())
large_groups.head()

## H3 Длинная, широкая и снова длинная форма

In [ ]:
long = decisions.groupby(long_keys).size().rename("n").reset_index()
wide = long.pivot(index=org_keys, columns="action_taken", values="n")
wide = wide.reindex(columns=[1, 2, 3], fill_value=0).fillna(0).astype("int64")
wide.columns.name = None
wide = wide.rename(columns={1:"issued", 2:"approved_not_accepted", 3:"denied"}).reset_index()
value_cols = ["issued", "approved_not_accepted", "denied"]
roundtrip = wide.melt(
    id_vars=org_keys, value_vars=value_cols,
    var_name="outcome", value_name="n"
)
roundtrip["action_taken"] = roundtrip["outcome"].map({
    "issued": 1, "approved_not_accepted": 2, "denied": 3
}).astype("int64")
roundtrip = roundtrip.loc[roundtrip["n"].gt(0), long_keys + ["n"]]
roundtrip = roundtrip.sort_values(long_keys).reset_index(drop=True)
expected = long.sort_values(long_keys).reset_index(drop=True)
assert_frame_equal(roundtrip, expected)
assert wide[value_cols].sum().sum() == len(decisions)
print("Round-trip совпал с исходной сводкой.")
print("Строк в исходной long-сводке:", len(long))
print("Строк после обратного преобразования:", len(roundtrip))
# Нули — отсутствующие сочетания; они удалены перед сравнением с исходной long-сводкой.

## H+ Безопасное соединение

In [ ]:
def safe_enrich(left, right):
    keys = org_keys
    if left[keys].isna().any().any() or right[keys].isna().any().any():
        raise ValueError("В ключах соединения есть пропуски")
    if right.duplicated(keys).any():
        raise ValueError("Ключи справочника повторяются")
    left_keys, right_keys = left[keys].drop_duplicates(), right[keys].drop_duplicates()
    missing_right = left_keys.merge(right_keys, on=keys, how="left", indicator=True)
    missing_left = right_keys.merge(left_keys, on=keys, how="left", indicator=True)
    if missing_right["_merge"].eq("left_only").any():
        raise ValueError("Для части решений нет записи в справочнике")
    if missing_left["_merge"].eq("left_only").any():
        raise ValueError("В справочнике есть ключи без решений")
    result = left.merge(right, on=keys, how="left", validate="many_to_one", indicator=True)
    if len(result) != len(left):
        raise AssertionError("JOIN изменил число строк")
    return result, result.loc[result["_merge"].eq("left_only")].copy()

safe_full, unmatched_full = safe_enrich(decisions, lookup)
assert len(safe_full) == len(decisions) and unmatched_full.empty
for bad_lookup in [
    lookup.iloc[1:].copy(),
    pd.concat([lookup, lookup.iloc[[0]]], ignore_index=True),
    pd.concat([lookup, lookup.iloc[[0]].assign(organization_name="Другое название")], ignore_index=True),
]:
    try:
        safe_enrich(decisions, bad_lookup)
    except ValueError:
        pass
    else:
        raise AssertionError("Некорректный справочник не был остановлен")
print("Полный справочник прошёл; частичный, повторный и конфликтный остановлены.")